# Big Data Analytics
Praktikum Sommersemester 2023. <small>Version 1.0</small>

**Aufgabe 3: Abfragen mit Apache Spark** 

Machen Sie sich mit Apache Spark vertraut. Bearbeiten Sie die Aufgaben indem Sie **Spark RDD** entsprechend transformieren. 

## Arbeitsanweisung
Nutzen Sie die markierten Zellen im vorliegenden Notebook `BDA1_A3_Spark.ipynb` für Ihre Lösungen und laden Sie es in Ilias hoch. In den Zellen muss ausführbarer python code vorliegen. Die Ausgabe soll unterhalb der jeweiligen Zellen produziert werden.
Liefern Sie auch aussagekräftiges Markdown zu Ihrem Code (Vorgehen, Quellen, etc) ab.

**Hinweis**: Verwenden Sie für diese Aufgaben *nicht* Spark SQL und *keine* Dataframes. 

----

## Vorbereitung
* Verwenden Sie immer den vorgegeben Spark Master um Inkonsistenzen der python3 Versionen zwischen Worker und Client zu verhindern.
* Ändern Sie nicht die SparkContext Konfiguration und beenden Sie bitte den SparkContext nachdem Sie die Bearbeitung beenden, um die Resourcen wieder frei zu geben! (`stop_sc1()`)
* Stellen Sie sicher, dass `pyspark` installiert ist (pip install)

In [25]:
!pip install pyspark

Für diese Aufgabe steht ein HDFS mit dem Namenode `namenode` unter Port `19000` bereit. 

Sie finden die folgenden Dateien darin:

* **`/data/bda1/co2data.tsv`**<br>Datensatz von Messungen verschiedener CO<sub>2</sub>-Sensoren
* **`/data/bda1/co2data_pm.tsv`**<br>Datensatz von Messungen verschiedener CO<sub>2</sub>-Sensoren mit Partikelmessung, pm ist die Partikeldichte, npm die Partikelanzahl bestimmter Größen

In [26]:
import pyspark
from pprint import pprint
import os
import sys

print(sys.version)  # python3 version

os.environ['PYSPARK_PYTHON'] = '/usr/bin/python3'
os.environ['PYSPARK_DRIVER_PYTHON'] = 'ipython3'
os.environ['PYSPARK_DRIVER_PYTHON_OPTS'] = 'notebook'
def stop_sc1():
    """stop spark context if exists"""
    try:
        sc1.stop()
        print('Spark Context stopped')
    except Exception as ex1:
        print(f'No context stopped: {ex1}')

3.9.6 | packaged by conda-forge | (default, Jul 11 2021, 03:39:48) 
[GCC 9.3.0]


### Spark Context erzeugen
Die Session mit dem vorgegebenen Spark Master öffnen.

In [27]:
stop_sc1()

# never ever change these lines!
config = pyspark.SparkConf().setAll([('spark.executor.memory', '1g'), ('spark.executor.cores', '1'), ('spark.cores.max', '2'), ('spark.driver.memory','1g'), ("spark.app.name", os.environ['JUPYTERHUB_CLIENT_ID'])])
sc1 = pyspark.SparkContext(master='spark://jupiter.bigdata.fh-aachen.de:17077', conf=config)

Spark Context stopped


## Aufgabe 3 a

Untersuchen Sie die Sensordaten unter ``hdfs://namenode/data/bda1/*``. Beantworten Sie die folgenden Fragen, indem Sie jeweils geeignete RDD Operationen in Spark ausführen:

1. Wieviele Messungen sind der Datenmenge `co2data.tsv` zu finden? Wieviele sind es in `co2data_pm.tsv` ?
2. Wie lauten die Attributnamen der Datenmenge `co2data_pm.tsv` ? Geben Sie sie zeilenweise aus.
3. Wieviele verschiedene Sensoren (angegeben im Feld _serial_number_) enhält die Datenmenge `co2data.tsv` ?  Beachten Sie den Hinweis unter 3a)
4. Wieviele Datenpunkte je Sensor liegen vor in `co2data.tsv` ? Geben Sie sowohl Sensor als auch Anzahl aus und beachten Sie den Hinweis unter 3a)
5. Was ist der höchste, und was der niedrigste Temperaturwert in der Datenmenge `co2data_pm.tsv` ? 
6. Was ist der durchschnittliche CO<sub>2</sub>-Wert je Sensor in der Datenmenge `co2data_pm.tsv` ? Runden Sie gerne auf drei Nachkommastellen und beachten Sie den Hinweis unter 3a)

**Hinweis:**<br>Die Serial Number besteht aus drei Komponenten: `s_` als Präfix, die eindeutige MAC-Adresse des Sensors und einer Zahl. Die Eindeutigkeit wird nur über die MAC-Adresse bestimmt. Nutzen Sie die MAC-Adresse.

## Durchführung Aufgabe 3a 1

Im folgenden Code werden zwei TSV-Dateien `co2data.tsv` und `co2data_pm.tsv` gelesen. Nachdem die zwei TSV-Dateien gelesen wurden, werden die RDDs `rdd_co2data` und `rdd_co2data_pm` erstellt, indem die Textdateien an den angegebenen Pfaden geladen werden.

Dann werden die Header-Zeilen der RDDs extrahiert, um sie später von den eigentlichen Daten zu trennen. Die Header-Zeile wird in den Variablen headerco2data und headerco2data_pm gespeichert.

Danach entferne ich bei beiden RDDs die Header-Zeilen, indem die erste Zeile rausgefiltert wird. Die erste Zeile enthält den Spaltennamen der jeweiligen Spalten, sodass diese bei der Auswertung der Aufgaben keine Rolle spielen dürfte. Die gefilterten RDDs werden in den Variablen `rdd_co2data_tmp` und `rdd_co2data_tmp_pm` gespeichert.

Im nächsten Schritt wird die Anzahl der Messungen in den RDDs gezählt, indem die `count()`-Funktion auf den jeweiligen RDDs angewendet wird. Die Ergebnisse werden in den Variablen `count_co2data` und `count_co2data_pm gespeichert`.

#### Quellen:

* https://stackoverflow.com/questions/27854919/how-do-i-skip-a-header-from-csv-files-in-spark
* https://spark.apache.org/docs/latest/rdd-programming-guide.html#initializing-spark
* https://spark.apache.org/docs/latest/api/python/reference/api/pyspark.RDD.count.html#pyspark.RDD.count

In [28]:
# Aufgabe 3a 1

path_co2data = "hdfs://namenode:19000/data/bda1/co2data.tsv"
path_co2data_pm = "hdfs://namenode:19000/data/bda1/co2data_pm.tsv"
rdd_co2data = sc1.textFile(path_co2data)
rdd_co2data_pm = sc1.textFile(path_co2data_pm)

headerco2data = rdd_co2data.first()
rdd_co2data_tmp = rdd_co2data.filter(lambda line: line != headerco2data)

headerco2data_pm = rdd_co2data_pm.first()
rdd_co2data_tmp_pm = rdd_co2data_pm.filter(lambda line: line != headerco2data_pm)

count_co2data = rdd_co2data_tmp.count()

count_co2data_pm = rdd_co2data_tmp_pm.count()


print("Anzahl der Messungen in co2data.tsv:", count_co2data)
print("Anzahl der Messungen in co2data_pm.tsv:", count_co2data_pm)

Anzahl der Messungen in co2data.tsv: 14013884
Anzahl der Messungen in co2data_pm.tsv: 14001337


## Durchführung Aufgabe 3a 2

Am Anfang wird die erste Zeile aus `rdd_co2data_pm` extrahiert. Durch die Extrahierung der ersten Zeile, ist es möglich die Zeilen mit der `split()`-Funktion zeilenweise auszugeben

#### Quellen:

* https://stackoverflow.com/questions/29753496/how-to-split-rows-of-a-spark-rdd-by-deliminator

In [33]:
# Aufgabe 3a 2

header = rdd_co2data_pm.first()

for attribute in header.split("\t"):
    print(attribute)

"timestamp"
"measurement_count"
"version"
"serial_number"
"co2_ppm"
"temperature_celsius"
"relative_humidity_percent"
"pm1"
"pm2"
"pm4"
"pm10"
"npm0"
"npm1"
"npm2"
"npm4"
"npm10"
"ps"


## Durchführung Aufgabe 3a 3

Dann wird eine Funktion extract_serial_number(x) definiert, um die MAC-Adresse aus der Seriennummer zu extrahieren. Die Funktion teilt die Zeile anhand des Tabulator-Zeichens ("\t") auf und extrahiert den Teil nach dem Unterstrich "_" im Feld serial_number als MAC-Adresse.

Das RDD rdd_co2data wird dann gefiltert, um die Zeilen zu entfernen, die der Header-Zeile entsprechen.

Anschließend wird die Funktion `extract_serial_number()` mit der `map()`-Transformation angewendet, um eine neue RDD `serial_numbers_mac` zu erstellen, die nur die extrahierten Seriennummern enthält.

Dann wird die Anzahl der verschiedenen Sensoren ermittelt, indem die `distinct()`-Funktion angewendet wird und anschließend die `count()`-Funktion auf das RDD.

#### Quellen:

* https://stackoverflow.com/questions/27854919/how-do-i-skip-a-header-from-csv-files-in-spark
* https://spark.apache.org/docs/latest/rdd-programming-guide.html#transformations
* https://www.w3schools.com/python/ref_string_split.asp
* https://spark.apache.org/docs/latest/api/python/reference/api/pyspark.RDD.count.html#pyspark.RDD.count
* https://spark.apache.org/docs/latest/rdd-programming-guide.html#passing-functions-to-spark
* https://spark.apache.org/docs/latest/api/python/reference/api/pyspark.RDD.distinct.html#pyspark.RDD.distinct

In [34]:
# Aufgabe 3a 3

def extract_serial_number(x):
    serial_number = x.split("\t")
    mac_address = serial_number[1].split("_")[1]
    return mac_address

header = rdd_co2data.first()
tmp_rdd_co2data = rdd_co2data.filter(lambda x: x != header)
serial_numbers_mac = tmp_rdd_co2data.map(extract_serial_number)
count_sensors = serial_numbers_mac.distinct().count()
print("Anzahl der verschiedenen Sensoren:", count_sensors)

Anzahl der verschiedenen Sensoren: 12


### Durchführung Aufgabe 3a 4

Der folgende Code Abschnitt zählt die Anzahl der Datenpunkte pro Sensor.

Zuerst wird die Header-Zeile des RDDs `rdd_co2data` in der Variable header gespeichert. Anschließend wird das RDD `co2_data_rdd` erstellt, indem die Header-Zeile aus rdd_co2data herausgefiltert wird.

Dann wird eine Funktion extract_mac_address(line) definiert, um die MAC-Adresse aus der Seriennummer zu extrahieren. Die Funktion teilt die Spalte anhand des Tabulator-Zeichens ("\t") auf und extrahiert den Teil nach dem Unterstrich "_" im Feld serial_number als MAC-Adresse.

Das RDD `mac_address_rdd` wird mit der `map()`-Transformation und der Funktion `extract_mac_address()` erstellt, um eine neue RDD `sensor_counts_rdd` zu erzeugen, die nur die extrahierten MAC-Adressen enthält.

Als nächstes wird das RDD verwendet, um die Anzahl der Datenpunkte pro Sensor zu zählen. Die `countByValue()`-Funktion wird auf das aktuell verwendete RDD angewendet und das Ergebnis wird anschließend ausgegeben.

#### Quellen:

* https://stackoverflow.com/questions/27854919/how-do-i-skip-a-header-from-csv-files-in-spark
* https://stackoverflow.com/questions/29753496/how-to-split-rows-of-a-spark-rdd-by-deliminator
* https://www.w3schools.com/python/ref_string_split.asp
* https://spark.apache.org/docs/latest/rdd-programming-guide.html#passing-functions-to-spark
* https://spark.apache.org/docs/latest/api/python/reference/api/pyspark.RDD.countByValue.html#pyspark.RDD.countByValue

In [41]:
# Aufgabe 3a 4

header = rdd_co2data.first()
co2_data_rdd = rdd_co2data.filter(lambda line: line != header)

def extract_mac_address(x):
    serial_number = x.split("\t")
    mac_address = serial_number[1].split("_")[1]
    return mac_address

mac_address_rdd = co2_data_rdd.map(extract_mac_address)

sensor_counts_rdd = mac_address_rdd.countByValue()

for sensor, anzahl_datenpunkte in sensor_counts_rdd.items():
    print("Sensor: {}, Anzahl: {}".format(sensor, anzahl_datenpunkte))

Sensor: d8bfc014724e, Anzahl: 2103522
Sensor: 10521c0202ab, Anzahl: 2064
Sensor: e8db84c5f771, Anzahl: 1665530
Sensor: 8caab57c3e19, Anzahl: 1561046
Sensor: e8db84c5f33d, Anzahl: 2270613
Sensor: 8caab57a6dd9, Anzahl: 2781666
Sensor: 10521c01cf19, Anzahl: 385105
Sensor: d8bfc0147061, Anzahl: 578457
Sensor: 3c6105d3abae, Anzahl: 1533919
Sensor: 8caab57cc961, Anzahl: 1131868
Sensor: e8db84c5f33d", Anzahl: 83
Sensor: 8caab57a6dd9", Anzahl: 11


### Durchführung Aufgabe 3a 5

Im folgenden Code wird die maximale und minimale Temperatur der TSV-Datei `co2data_pm.tsv` ausgegeben.

Zu Beginn wird wie jede Teilaufgabe zuvor die erste Zeile extrahiert, sodass diese herausgefiltert werden kann.

Es wird die Funktion `extract_temperature(line)` definiert, um die Spalte `temperature_celsius` zu extrahieren. Die Spalte wird anhand des Tabulator-Zeichens ("\t") aufgeteilt und der Wert `value[5]` an der Spalte `temperature_celsius` wird als Temperatur betrachtet. Dabei wird das doppelte Anführungszeichen mit der `replace()`-Funktion durch einen leeren String ersetzt. Dieser Schritt ist nötig, damit die String-Werte erfolgreich in Float-Werte konvertiert werden können. Falls der Wert nicht in eine Gleitkommazahl umgewandelt werden kann, wird `None` zurückgegeben. Ansonsten wird die Temperatur zurückgegeben, nachdem erfolgreich konvertiert wurde.

Das RDD `tmp_rdd_co2data_pm` wird verwendet, um die Funktion `extract_temperature()` anzuwenden und eine neue RDD `filtered_temperatures` zu erstellen. Dabei werden alle `None`-Werte herausgefiltert.

Anschließend wird der höchste und niedrigste Temperaturwert mit den Funktionen `max()` und `min()` berechnet und in den Variablen `max_temperature` und `min_temperature` gespeichert und ausgegeben.

#### Quellen:

* https://stackoverflow.com/questions/29753496/how-to-split-rows-of-a-spark-rdd-by-deliminator
* https://stackoverflow.com/questions/46957801/how-to-replace-remove-regular-expression-in-pyspark-rdd
* https://stackoverflow.com/questions/27854919/how-do-i-skip-a-header-from-csv-files-in-spark
* https://spark.apache.org/docs/latest/rdd-programming-guide.html#passing-functions-to-spark
* https://spark.apache.org/docs/latest/api/python/reference/api/pyspark.RDD.min.html#pyspark.RDD.min
* https://spark.apache.org/docs/latest/api/python/reference/api/pyspark.RDD.max.html#pyspark.RDD.max

In [36]:
# Aufgabe 3a 5

def extract_temperature(x):
    temperature_celsius = x.split("\t")
    try:
        temperature = float(temperature_celsius[5].replace('"', ''))
    except ValueError:
        return None
    return temperature

first_lines = rdd_co2data_pm.first()
tmp_rdd_co2data_pm = rdd_co2data_pm.filter(lambda x: x != first_lines)

filtered_temperatures = tmp_rdd_co2data_pm.map(extract_temperature).filter(lambda x: x is not None)

max_temperature = filtered_temperatures.max()
min_temperature = filtered_temperatures.min()

print("Höchster Temperaturwert: {}".format(max_temperature))
print("Niedrigster Temperaturwert: {}".format(min_temperature))

Höchster Temperaturwert: 41.0
Niedrigster Temperaturwert: 6.0


### Durchführung Aufgabe 3a 6

Am Anfang wird die erste Zeile wieder extrahiert um sie herausfiltern zu können.

Zuerst wird eine Funktion `extract_mac_co2(line)` definiert, um die MAC-Adresse und den CO2-Wert zu extrahieren. Die Spalten werden anhand des Tabulator-Zeichens ("\t") aufgeteilt, und die MAC-Adresse wird aus dem Teil nach dem Unterstrich "_" im Feld serial_numbers extrahiert. Der CO2-Wert wird in einen Float konvertiert. Dabei wwerden doppelte Anführungszeichen mithilfe der `replace()`-Funktion durch leere Strings ersetzt, um Konvertierungsschwierigkeiten zu vermeiden. Falls der CO2-Wert nicht in eine Gleitkommazahl umgewandelt werden kann, wird `None` zurückgegeben. Ansonsten wird ein Tupel mit der MAC-Adresse und dem CO2-Wert zurückgegeben.

Die Funktion `avg_co2` berechnet den durchschnittlichen CO2-Wert je Sensor. Der CO2-Wert wird auf drei Nachkommastellen gerundet

Das RDD `tmp_rdd_co2data_pm` wird verwendet, um die Funktion `extract_mac_co2()` anzuwenden und eine neue RDD `mac_co2_rdd` zu erstellen. Dabei werden alle None-Werte herausgefiltert.

Anschließend wird das RDD gruppiert nach der MAC-Adresse und der Durchschnitt der CO2-Werte pro Sensor wird mit der `groupByKey()`-Funktion und der `mapValues()`-Funktion berechnet.
Bei der `groupByKey()`-Funktion gruppieren wir die CO2-Werte für jede MAC-Adresse und mit der `mapValues()`-Funktion übergeben wir jeden CO2-Wert im Key-Value Paar der Form (MAC-Adresse, CO2-Wert) und berechnen den Durchschnittswert pro Sensor. 

Dann werden die Daten aus `sensor_avg_co2_rdd` mit der collect() abgerufen und ausgegeben.

#### Quellen:

* https://stackoverflow.com/questions/29753496/how-to-split-rows-of-a-spark-rdd-by-deliminator
* https://stackoverflow.com/questions/46957801/how-to-replace-remove-regular-expression-in-pyspark-rdd
* https://stackoverflow.com/questions/27854919/how-do-i-skip-a-header-from-csv-files-in-spark
* https://www.w3schools.com/python/ref_string_split.asp
* https://spark.apache.org/docs/latest/rdd-programming-guide.html#passing-functions-to-spark
* https://spark.apache.org/docs/latest/api/python/reference/api/pyspark.RDD.groupByKey.html#pyspark.RDD.groupByKey

In [42]:
# Aufgabe 3a 6

def extract_mac_co2(x):
    values = x.split("\t")
    mac_address = values[3].split("_")[1]
    try:
        float_CO2 = float(values[4].replace('"', ''))
    except ValueError:
        return None
    return(mac_address, float_CO2)

def avg_co2(values):
    return round(sum(values) / len(values), 3)

first_lines = rdd_co2data_pm.first()
tmp_rdd_co2data_pm = rdd_co2data_pm.filter(lambda x: x != first_lines)   

mac_co2_rdd = tmp_rdd_co2data_pm.map(extract_mac_co2).filter(lambda values: values is not None)

sensor_avg_co2_rdd = mac_co2_rdd.groupByKey().mapValues(avg_co2).collect()

for sensor, co2_avg in sensor_avg_co2_rdd:
    print("Sensor: {}, Durchschnittlicher CO2-Wert: {}".format(sensor, co2_avg))

Sensor: e8db84c5fc6a, Durchschnittlicher CO2-Wert: 554.979
Sensor: 308398a2a1f6, Durchschnittlicher CO2-Wert: 657.518
Sensor: 8caab57c9751, Durchschnittlicher CO2-Wert: 519.263
Sensor: 8caab57cc813, Durchschnittlicher CO2-Wert: 547.52
Sensor: 8caab57d01da, Durchschnittlicher CO2-Wert: 574.967
Sensor: 308398a2ddcb, Durchschnittlicher CO2-Wert: 481.186
Sensor: 3c6105d3908f, Durchschnittlicher CO2-Wert: 730.933
Sensor: 8caab57cbb52, Durchschnittlicher CO2-Wert: 496.846
Sensor: a848fac03782, Durchschnittlicher CO2-Wert: 549.538
Sensor: ac0bfbd6547d, Durchschnittlicher CO2-Wert: 486.608
Sensor: 308398b5b69c, Durchschnittlicher CO2-Wert: 1058.714
Sensor: 308398a2fb52, Durchschnittlicher CO2-Wert: 541.715
Sensor: 308398a2f790, Durchschnittlicher CO2-Wert: 688.314
Sensor: ac0bfbd64321, Durchschnittlicher CO2-Wert: 580.367
Sensor: 308398b595c0, Durchschnittlicher CO2-Wert: 531.905
Sensor: e8db84c62ab4, Durchschnittlicher CO2-Wert: 560.562
Sensor: 3c6105d381e8, Durchschnittlicher CO2-Wert: 546.0

## Aufgabe 3 b

Machen Sie Aussagen in Markdown zu den beiden Datenmengen, nachdem Sie sich die Information aus Spark gezogen haben.

* Kommen Sensoren (MAC-Adresse in Serial Number) in beiden Datenmengen vor? Wenn ja, welche?
* Anhand welchen Attributs können Sie auf vorhandene Werte in den Attributen `pm*` und `npm*` filtern? Geben Sie ein Beispiel.
* Enthält eine Datenmenge "fehlerhafte" CO<sub>2</sub>-Messungen? Wenn ja, wieviele Messungen sind betroffen? Fehlerhaft ist ein Wert `null`


## Durchführung Aufgabe 3b 1

Im Folgendem Code wird überprüft, ob Sensoren in beide Datenmengen `co2data.tsv` und `co2data_pm.tsv` vorkommen.

Dazu wird zu Beginn, wie in den vorherigen Aufgaben die Spaltennamen rausgefiltert, da diese für die Auswertung keine Rolle spielt.

In den Funktionen `extract_serial_num` und `extract_serial_num_pm` werden die Felder `serial_number` extrahiert und so angepasst, sodass die MAC-Adresse verwendet wird.

Die RDDs `mac_address_rdd1` und `mac_address_rd_pm` werden erstellt, indem die entsprechenden Funktionen auf die RDDs `rdd_co2data_tmp` und `rdd_co2data_tmp_pm` angewendet werden. Mit der `distinct()`-Transformation werden doppelte MAC-Adressen entfernt.

Dann wird die `intersection()`-Transformation verwendet, um die gemeinsamen MAC-Adressen aus beiden Datenmengen zu ermitteln.

### Quellen:

* siehe oben vorherige Aufgaben

## Auswertung Aufgabe 3b 1

In beiden Datenmengen `co2data.tsv` und `co2data_pm.tsv` kommen die Sensoren 'e8db84c5f33d', 'e8db84c5f771', '10521c0202ab', '8caab57c3e19', '3c6105d3abae' vor.

In [38]:
# keine Zellenvorgabe

headerco2data = rdd_co2data.first()
rdd_co2data_tmp = rdd_co2data.filter(lambda line: line != headerco2data)

headerco2data_pm = rdd_co2data_pm.first()
rdd_co2data_tmp_pm = rdd_co2data_pm.filter(lambda line: line != headerco2data_pm)

def extract_serial_num(x):
    serial_number = x.split("\t")[1]
    mac_address = serial_number.split("_")[1]
    return mac_address
    
def extract_serial_num_pm(x):
    serial_number = x.split("\t")[3]
    mac_address = serial_number.split("_")[1]
    return mac_address

mac_address_rdd1 = rdd_co2data_tmp.map(extract_serial_num).distinct()
mac_address_rd_pm = rdd_co2data_tmp_pm.map(extract_serial_num_pm).distinct()

common_mac_addresses = mac_address_rdd1.intersection(mac_address_rd_pm)

common_mac_addresses.collect()

['e8db84c5f33d',
 'e8db84c5f771',
 '10521c0202ab',
 '8caab57c3e19',
 '3c6105d3abae']

Auswertung Aufgabe 3b 2

In dieser Aufgabe sollen wir untersuchen anhand welchen Attributs man auf vorhandene Werte in den Attributen pm* und npm* filtern kann. Dazu habe ich mir pm1 und npm1 ausgeben lassen und mir ist aufgefallen, dass nur `null`-Werte ausgegeben werden. Dann habe ich die Atrribute `pm1` und `npm1` nach ungleich `null` filtern lassen. Als Ausgabe erhalte ich normale Werte und keine `null`-Werte. Ich habe die Aufgabe zumindestens so verstanden, damit ich vorhandene Werte als Ausgabe erhalten kann, muss ich die Attribute filtern, indem ich die `null` auslasse bei der Ausgabe.

In [39]:
print_rdd = rdd_co2data_pm.filter(lambda x: x.split("\t")[7] != '"null"' and x.split("\t")[11] != '"null"').\
map(lambda x: (x.split("\t")[7], x.split("\t")[11]))

print_rdd.collect()

[('"pm1"', '"npm0"'),
 ('"1.58"', '"10.74"'),
 ('"2.12"', '"14.56"'),
 ('"1.59"', '"10.8"'),
 ('"2.15"', '"14.78"'),
 ('"1.69"', '"11.45"'),
 ('"2.1"', '"14.45"'),
 ('"1.77"', '"12.04"'),
 ('"2.01"', '"13.78"'),
 ('"1.78"', '"12.07"'),
 ('"1.93"', '"13.26"'),
 ('"1.7"', '"11.56"'),
 ('"1.88"', '"12.89"'),
 ('"1.51"', '"10.3"'),
 ('"1.78"', '"12.2"'),
 ('"1.42"', '"9.65"'),
 ('"1.69"', '"11.62"'),
 ('"1.48"', '"10.1"'),
 ('"1.84"', '"12.61"'),
 ('"1.5"', '"10.21"'),
 ('"2.08"', '"14.27"'),
 ('"1.42"', '"9.68"'),
 ('"2.22"', '"15.22"'),
 ('"1.46"', '"9.95"'),
 ('"2.21"', '"15.21"'),
 ('"1.49"', '"10.16"'),
 ('"2.24"', '"15.38"'),
 ('"1.41"', '"9.59"'),
 ('"2.34"', '"16.08"'),
 ('"1.36"', '"9.24"'),
 ('"2.46"', '"16.87"'),
 ('"1.36"', '"9.26"'),
 ('"2.59"', '"17.77"'),
 ('"1.48"', '"10.03"'),
 ('"2.6"', '"17.86"'),
 ('"1.56"', '"10.63"'),
 ('"1.53"', '"10.37"'),
 ('"2.47"', '"16.96"'),
 ('"1.5"', '"10.15"'),
 ('"2.45"', '"16.81"'),
 ('"1.58"', '"10.75"'),
 ('"2.39"', '"16.41"'),
 ('"1.73"

## Durchführung Aufgabe 3b 3

In diesem Code Abschnitt wird überprüft ob es fehlerhafte CO2-Messungen gibt und falls es sie gibt, wie viele davon betroffen sind.

Dazu wird zu Beginn, wie in den vorherigen Aufgaben die Spaltennamen rausgefiltert, da diese für die Auswertung keine Rolle spielt.

Dann werden zwei Filteroperationen durchgeführt, um nur diejenigen Zeilen beizubehalten, die Null-Werte enthalten. Dazu werden die Zeilen von der Spalte serial_number aus den beiden Datenmengen `co2data.tsv` und `co2data_pm.tsv` gefiltert.

Die Anzahl der betroffenen Messungen wird mit der `count()`-Aktion ermittelt und in den Variablen `num_errors` und `num_errors_pm` gespeichert.

### Quellen:

* siehe oben vorherige Aufgaben

## Auswertung Aufgabe 3b 3

Bei der Asuwertung sieht man, dass in der Datenmenge `co2data.tsv` 19 `null`-Werte enthalten sind und in der Datenmenge `co2data_pm.tsv` sind 9 `null`-Werte enthalten. Die `null`-Werte weisen auf die fehlerhaften CO2-Messungen hin.

In [40]:
headerco2data = rdd_co2data.first()
rdd_co2data_tmp = rdd_co2data.filter(lambda line: line != headerco2data)

headerco2data_pm = rdd_co2data_pm.first()
rdd_co2data_tmp_pm = rdd_co2data_pm.filter(lambda line: line != headerco2data_pm)

error_data_rdd = rdd_co2data_tmp.filter(lambda x: x.split("\t")[3] == '"null"')

error_data_rdd_pm = rdd_co2data_tmp_pm.filter(lambda x: x.split("\t")[4] == '"null"')

num_errors = error_data_rdd.count()
num_errors_pm = error_data_rdd_pm.count()

print("Anzahl der fehlerhaften Messungen in co2data.tsv:", num_errors)
print("Anzahl der fehlerhaften Messungen in co2data.tsv_pm:", num_errors_pm)

Anzahl der fehlerhaften Messungen in co2data.tsv: 19
Anzahl der fehlerhaften Messungen in co2data.tsv_pm: 9


In [24]:
stop_sc1()  # always exit your spark context after work!

Spark Context stopped


## Nützliche Links
* https://spark.apache.org/docs/latest/rdd-programming-guide.html
* https://spark.apache.org/docs/latest/api/python/reference/pyspark.html#rdd-apis
* https://spark.apache.org/examples.html